# T10 It1: evaluation fixes before any new experiment

a. **Jetson context bug:** zgap / ligap were computed from all 10 stored candidates per query, while the Jetson replay matches only rank < 5.
   Fixed in `candidate_judge.add_context(table, m=5)`; the NUDT-only judge's Jetson numbers are redone with context from the 5 matched candidates.
b. **Empty shortlist:** `decide` now returns a skip for every photo that has no usable candidate (also when M = 5 leaves none), never an
   exception or a missing row (unit test `test_iteration10_blocks.py`).
c. **Failed calibration:** `ltt_tau` returns None when no cut passes, and the rule then answers nothing (`NO_ANSWER`, the override included);
   before, it fell back to the strictest cut (unit test).
d. **Configuration manifest** saved with every result (block `train/checks/eval_tools.py`), compared with the submitted headless settings.
e. **Failure log per photo:** never retrieved / retrieved but unmatched (< 3 inliers) / retrieved but rejected / answered off / wrong / right.

In [ ]:
import os, sys, time, json
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
import train.checks.candidate_judge as cj
from train.checks.holdout import NUDT_DESIGN, load_t8_table
from train.checks.replay import paired_bootstrap
from train.checks.eval_tools import compare_with_headless, failure_log, manifest, save_manifest
RES = ROOT / "tests/Test_Iteration_10/results"
T10 = load_t8_table(ROOT, locked=False, blind=True)        # T8 final sets are seen data now; nothing here is tuned on them
T5 = cj.add_context(T10, m=5)
SETS = ["official_try", "official_test", "nudt707"] + NUDT_DESIGN
T10.groupby("data").photo.nunique()

## a. Jetson numbers with context from the 5 matched candidates (NUDT-only judge, early exit 0.98)

In [ ]:
K = dict(override=12, min_choice=3, wide_tau=0.95, jetson=True, m=5, early_tau=0.98)
p10, model10 = cj.judge_probability(T10, NUDT_DESIGN, cj.FEATURES_CTX, "gbm")
p_a = np.zeros(len(T5)); u = (T5.inliers >= 3).to_numpy(); p_a[u] = model10.predict_proba(T5.loc[u, cj.FEATURES_CTX].to_numpy(float))[:, 1]
p_b, _ = cj.judge_probability(T5, NUDT_DESIGN, cj.FEATURES_CTX, "gbm")
d5 = T5.data.isin(NUDT_DESIGN).to_numpy()
tau_b = cj.ltt_tau(cj.scan_tau(T5[d5], p_b[d5], cj.TAUS, **{k: v for k, v in K.items() if k != "early_tau"}), NUDT_DESIGN, 0.005)
tau_b = cj.NO_ANSWER if tau_b is None else max(tau_b, 0.9); print("Learn-then-Test cut with M = 5 context:", tau_b)
base = cj.baseline(T10, 8, 8, jetson=True, m=5); b = base.set_index(["data", "photo"]).pts
RUNS = {"T8 (context from 10, the bug)": cj.decide(T10, p10, 0.925, **K),
        "context from 5, judge trained on 10-context": cj.decide(T5, p_a, 0.925, **K),
        "context from 5, judge trained on 5-context (fixed)": cj.decide(T5, p_b, tau_b, **K)}
rows = []
for n, r in RUNS.items():
    s = cj.summary(r); rr = r.set_index(["data", "photo"]).pts
    for d in SETS:
        g = paired_bootstrap(b.loc[d].to_numpy(), rr.loc[d].reindex(b.loc[d].index).to_numpy())
        rows.append(dict(rule=n, data=d, score=s.loc[d, "score"], wrong=int(s.loc[d, "wrong"]), queries=s.loc[d, "queries"], gain="%+.2f [%+.2f, %+.2f]" % g))
CTX = pd.DataFrame(rows); CTX.to_csv(RES / "it1_jetson_context_fix.csv", index=False)
save_manifest(RES / "it1_jetson_context_fix.csv", manifest("jetson", cut=tau_b, wide_cut=0.95, override=12, early_exit=0.98,
              judge_features=cj.FEATURES_CTX, judge_trained_on=NUDT_DESIGN, calibration="Learn-then-Test alpha 0.005 on NUDT design, floor 0.9",
              note="context (zgap, ligap) from rank < 5"))
CTX.pivot(index="data", columns="rule", values="score").round(2).join(CTX.pivot(index="data", columns="rule", values="wrong"), rsuffix=" wrong")

In [ ]:
CTX[CTX.data.isin(["official_test", "nudt707", "official_try"])].set_index(["data", "rule"])

## b-c. Every photo gets a row; a failed calibration answers nothing

In [ ]:
n_photos = T10.groupby("data").photo.nunique()
for name, r in RUNS.items():
    assert (r.groupby("data").photo.nunique() == n_photos).all(), name
scan = cj.scan_tau(T5[d5], p_b[d5], cj.TAUS, **{k: v for k, v in K.items() if k != "early_tau"})
print("LTT at alpha 0.0001 (impossible on this much data):", cj.ltt_tau(scan, NUDT_DESIGN, 0.0001))
r = cj.decide(T5, p_b, cj.NO_ANSWER, **K); print("answers with NO_ANSWER:", int((r.stage != "skip").sum()), "of", len(r))

## d. Configuration manifest vs the submitted headless pipeline

In [ ]:
for name, m in {"PC research replay (judge)": manifest("pc", cut=0.9, wide_cut=0.95, override=12, judge_features="FEATURES_CTX", judge_trained_on=NUDT_DESIGN),
                "Jetson research replay (judge + early exit)": manifest("jetson", cut=tau_b, wide_cut=0.95, override=12, early_exit=0.98, judge_features="FEATURES_CTX", judge_trained_on=NUDT_DESIGN),
                "Jetson today replay (bars)": manifest("jetson", bars=[8, 8], min_choice=1)}.items():
    print("==", name); display(compare_with_headless(m))

**Read:** the replays differ from the submission in ways that matter for comparing numbers: (1) the submission's Jetson run used the true
position as the hint, the research replays a 5 km random hint (harder); (2) the research Jetson replay caps the queries at 10 to model the
22 s budget, the submission is cut by time; (3) the PC replay's 27 queries include scale 0.95, the submission's 18 do not; (4) the matcher
confidence: research keeps matches >= 0.22 with a LoFTR coarse threshold 0.21, the Notion page lists one value, 0.21: which parameter it is
must be checked in the headless code (not on this machine).

## e. Failure log per photo

In [ ]:
RULES = {("pc", "bar 8 / 11"): (T10, cj.baseline(T10, 8, 11, min_choice=3)),
         ("pc", "judge (NUDT only), 0.9 + override 12"): (T10, cj.decide(T10, p10, 0.9, override=12, min_choice=3, wide_tau=0.95)),
         ("jetson", "Jetson today"): (T10, base),
         ("jetson", "judge (NUDT only, 5-context) + early exit"): (T5, RUNS["context from 5, judge trained on 5-context (fixed)"])}
parts = []
for (var, name), (t, r) in RULES.items():
    parts.append(failure_log(t, r, var).assign(var=var, rule=name))
FAIL = pd.concat(parts); FAIL.to_csv(RES / "it1_failure_log.csv", index=False)
order = ["answered_right", "answered_off", "answered_wrong", "retrieved_but_rejected", "retrieved_but_unmatched", "never_retrieved"]
FAIL.groupby(["data", "var", "rule"]).outcome.value_counts().unstack(fill_value=0).reindex(columns=order, fill_value=0)

**Read:** "never retrieved" is a coarse-search / DINO loss no decision rule can recover; "retrieved but unmatched" is a fine-matching
loss (the right place was among the candidates but got < 3 inliers); "retrieved but rejected" is the decision's own loss.